# Notebook 08 — Peta Prioritas Desa (Hasil Akhir)

**Tujuan:** Menggabungkan seluruh hasil menjadi satu keluaran akhir yang siap dipakai
Dinas Sosial: **daftar & peta prioritas desa** dengan zona kemiskinan, skor prioritas,
jenis masalah dominan, dan rekomendasi intervensi.

**Menyatukan:**
- Zona kemiskinan → dari **clustering K-Means** (NB03)
- Skor & peringkat prioritas → dari **TOPSIS + bobot AHP** (NB06/NB07)
- Jenis masalah & rekomendasi → aturan berbasis profil (ekonomi vs administratif)

**Data:** 20 Juli 2026, 371 desa. **Output:** `output/spk/peta_prioritas/`.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SPK_DIR = ROOT / 'output' / 'spk'
OUT_DIR = SPK_DIR / 'peta_prioritas'
OUT_DIR.mkdir(parents=True, exist_ok=True)
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10})

# Sumber tunggal: hasil TOPSIS+AHP (sudah memuat klaster & fitur)
df = pd.read_csv(SPK_DIR / 'ranking_prioritas_desa_AHP.csv')
print(f'Jumlah desa : {len(df)}')
print('Kolom       :', list(df.columns))

Jumlah desa : 371
Kolom       : ['KECAMATAN', 'KELURAHAN', 'JUMLAH_KELUARGA', 'pct_desil1', 'pct_desil2', 'pct_desil3', 'pct_desil4', 'pct_desil5', 'pct_desil6_10', 'pct_belum_prmk', 'pct_nonaktif', 'cluster_kmeans', 'skor_topsis_ahp', 'rank_topsis_ahp', 'skor_saw_ahp', 'rank_saw_ahp', 'rank_topsis']


## 1. Beri Nama Zona Klaster

Klaster K-Means (0/1/2) diberi label berdasarkan rata-rata proporsi keluarga sangat miskin
(Desil 1 + Desil 2): klaster dengan nilai tertinggi = *Kemiskinan Tinggi*, dst.

In [2]:
prof = df.groupby('cluster_kmeans').apply(
    lambda g: (g['pct_desil1'] + g['pct_desil2']).mean()
).sort_values(ascending=False)

nama_zona = {}
label_urut = ['Kemiskinan Tinggi', 'Kemiskinan Sedang', 'Kemiskinan Rendah']
for i, cl in enumerate(prof.index):
    nama_zona[cl] = label_urut[i]

df['zona'] = df['cluster_kmeans'].map(nama_zona)
print('Pemetaan klaster -> zona (urut Desil1+2 tertinggi):')
for cl in prof.index:
    print(f'  Klaster {cl}  (Desil1+2 rata2 {prof[cl]:.1f}%)  ->  {nama_zona[cl]}')
print()
print(df['zona'].value_counts().to_string())

Pemetaan klaster -> zona (urut Desil1+2 tertinggi):
  Klaster 0  (Desil1+2 rata2 34.5%)  ->  Kemiskinan Tinggi
  Klaster 2  (Desil1+2 rata2 33.5%)  ->  Kemiskinan Sedang
  Klaster 1  (Desil1+2 rata2 18.2%)  ->  Kemiskinan Rendah

zona
Kemiskinan Tinggi    136
Kemiskinan Rendah    126
Kemiskinan Sedang    109


~\AppData\Local\Temp\ipykernel_2880\1145458583.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prof = df.groupby('cluster_kmeans').apply(


## 2. Tentukan Jenis Masalah Dominan

Tiap desa dinilai: apakah masalahnya **ekonomi** (kemiskinan desil) atau **administratif/cakupan**
(masih banyak keluarga belum diperingkat sehingga belum bisa menerima bantuan).

**Revisi:** penanda administratif kini **hanya** memakai proporsi *belum diperingkat*.
Kriteria *nonaktif* tidak lagi dipakai — konsisten dengan keputusan mengeluarkannya dari
pembobotan AHP, karena status nonaktif dapat timbul dari sebab wajar (keluarga sudah mampu,
pindah domisili, atau meninggal). Ambang: proporsi berada di 25% teratas se-kabupaten.

In [3]:
# REVISI: kriteria 'Nonaktif' tidak lagi dipakai sebagai penentu jenis masalah,
# konsisten dengan keputusan mengeluarkannya dari pembobotan AHP (NB07).
# Penanda masalah administratif kini hanya berdasarkan proporsi keluarga
# BELUM DIPERINGKAT, yaitu indikator langsung bahwa masih ada keluarga yang
# belum masuk sistem pemeringkatan DTSEN sehingga tidak dapat menerima bantuan.
amb_belum = df['pct_belum_prmk'].quantile(0.75)
print(f'Ambang belum-PRMK (persentil 75) : {amb_belum:.2f}%')

def jenis_masalah(r):
    if r['pct_belum_prmk'] >= amb_belum:
        return 'Administratif/Cakupan'
    return 'Ekonomi'

df['jenis_masalah'] = df.apply(jenis_masalah, axis=1)
print()
print(df['jenis_masalah'].value_counts().to_string())

Ambang belum-PRMK (persentil 75) : 4.64%

jenis_masalah
Ekonomi                  278
Administratif/Cakupan     93


## 3. Rekomendasi Intervensi

Kombinasi **zona** + **jenis masalah** menghasilkan rekomendasi tindakan.

In [4]:
def rekomendasi(r):
    if r['jenis_masalah'] == 'Administratif/Cakupan':
        return 'Prioritas survei/pemutakhiran data (musdes)'
    if r['zona'] == 'Kemiskinan Tinggi':
        return 'Prioritas bantuan tunai (PKH/BLT)'
    if r['zona'] == 'Kemiskinan Sedang':
        return 'Bantuan pangan + pemberdayaan ekonomi mikro'
    return 'Bukan prioritas / pemberdayaan preventif'

df['rekomendasi'] = df.apply(rekomendasi, axis=1)
print(df.groupby(['zona', 'jenis_masalah']).size().to_string())

zona               jenis_masalah        
Kemiskinan Rendah  Administratif/Cakupan     39
                   Ekonomi                   87
Kemiskinan Sedang  Administratif/Cakupan      8
                   Ekonomi                  101
Kemiskinan Tinggi  Administratif/Cakupan     46
                   Ekonomi                   90


## 4. Tabel Prioritas Final

In [5]:
df = df.sort_values('rank_topsis_ahp').reset_index(drop=True)
df['prioritas'] = df['rank_topsis_ahp']

final = df[['prioritas', 'KECAMATAN', 'KELURAHAN', 'zona',
            'pct_desil1', 'pct_desil2', 'pct_belum_prmk', 'pct_nonaktif',
            'skor_topsis_ahp', 'jenis_masalah', 'rekomendasi']].copy()
final.columns = ['Prioritas', 'Kecamatan', 'Desa', 'Zona',
                 '%Desil1', '%Desil2', '%BelumData', '%Nonaktif',
                 'Skor', 'Jenis Masalah', 'Rekomendasi']
final.to_csv(OUT_DIR / 'peta_prioritas_final.csv', index=False)

print('=== 15 DESA PRIORITAS TERATAS ===')
print(final.head(15).round(3).to_string(index=False))

=== 15 DESA PRIORITAS TERATAS ===
 Prioritas          Kecamatan              Desa              Zona  %Desil1  %Desil2  %BelumData  %Nonaktif  Skor         Jenis Masalah                                 Rekomendasi
         1      Kikim Selatan  Beringin Janggut Kemiskinan Tinggi   44.366   13.380       4.930      9.155 0.613 Administratif/Cakupan Prioritas survei/pemutakhiran data (musdes)
         2        Muarapayang       Bandu Agung Kemiskinan Tinggi   39.437   18.779       4.225      9.624 0.601               Ekonomi           Prioritas bantuan tunai (PKH/BLT)
         3              Jarai       Lubuk Saung Kemiskinan Tinggi   30.576   15.827       5.396     12.590 0.543 Administratif/Cakupan Prioritas survei/pemutakhiran data (musdes)
         4       Tanjungtebat     Padang Perigi Kemiskinan Tinggi   30.290   19.917       4.149      9.959 0.520               Ekonomi           Prioritas bantuan tunai (PKH/BLT)
         5      Kikim Selatan        Keban Raya Kemiskinan Tinggi   24.

## 5. Visualisasi

In [6]:
# Gambar 1: 15 desa prioritas teratas (bar horizontal, warna per jenis masalah)
top = df.head(15).iloc[::-1]
warna = top['jenis_masalah'].map({'Ekonomi': '#C0392B', 'Administratif/Cakupan': '#2E86C1'})
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top['KECAMATAN'] + ' - ' + top['KELURAHAN'], top['skor_topsis_ahp'], color=warna)
ax.set_xlabel('Skor Prioritas (TOPSIS + bobot AHP)')
ax.set_title('15 Desa Prioritas Teratas — Kab. Lahat (Data 20 Juli 2026)', fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#C0392B', label='Masalah Ekonomi'),
                   Patch(color='#2E86C1', label='Masalah Administratif')], loc='lower right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'top15_desa_prioritas.png', dpi=150, bbox_inches='tight')
plt.close()
print('Disimpan: top15_desa_prioritas.png')

Disimpan: top15_desa_prioritas.png


In [7]:
# Gambar 2: rata-rata skor prioritas per kecamatan (proxy 'peta' wilayah)
kec = df.groupby('KECAMATAN')['skor_topsis_ahp'].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
colors = plt.cm.RdYlGn_r((kec - kec.min()) / (kec.max() - kec.min()))
ax.barh(kec.index, kec.values, color=colors)
ax.set_xlabel('Rata-rata Skor Prioritas Desa')
ax.set_title('Prioritas Antar-Kecamatan (rata-rata skor desa)\nMerah = lebih prioritas',
             fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'prioritas_per_kecamatan.png', dpi=150, bbox_inches='tight')
plt.close()
print('Disimpan: prioritas_per_kecamatan.png')
print('\n5 kecamatan paling prioritas (rata-rata skor tertinggi):')
print(kec.sort_values(ascending=False).head(5).round(4).to_string())

Disimpan: prioritas_per_kecamatan.png

5 kecamatan paling prioritas (rata-rata skor tertinggi):
KECAMATAN
Muarapayang       0.4713
Pseksu            0.4080
Kikim Selatan     0.4018
Merapi Selatan    0.3937
Jarai             0.3673


## 6. Catatan Anomali Data

Desa **Talang Jawa (Kec. Lahat)** dikeluarkan dari analisis karena hanya memiliki 1 keluarga
yang 100% belum diperingkat (tanpa profil kemiskinan). Desa ini **tidak diabaikan**, melainkan
ditandai sebagai temuan tersendiri: **memerlukan verifikasi/survei lapangan segera** karena
mengindikasikan data DTSEN desa tersebut belum lengkap.

In [8]:
anomali = pd.DataFrame([{
    'Kecamatan': 'Lahat', 'Desa': 'Talang Jawa',
    'Keterangan': '1 KK, 100% belum diperingkat — tanpa profil kemiskinan',
    'Rekomendasi': 'Verifikasi & pemutakhiran data lapangan (prioritas administratif)',
}])
anomali.to_csv(OUT_DIR / 'anomali_data.csv', index=False)

ringkas = {
    'sumber_data': '20 Juli 2026',
    'jumlah_desa_dianalisis': int(len(df)),
    'distribusi_zona': df['zona'].value_counts().to_dict(),
    'distribusi_jenis_masalah': df['jenis_masalah'].value_counts().to_dict(),
    'top5_desa_prioritas': df.head(5)[['KECAMATAN', 'KELURAHAN', 'skor_topsis_ahp']].to_dict('records'),
    'anomali': 'Talang Jawa (Kec. Lahat) — perlu survei lapangan',
}
with open(OUT_DIR / 'ringkasan_peta_prioritas.json', 'w', encoding='utf-8') as f:
    json.dump(ringkas, f, indent=2, ensure_ascii=False)
print('Semua output final di:', OUT_DIR)
print(json.dumps(ringkas, indent=2, ensure_ascii=False))

Semua output final di: analysis\output\spk\peta_prioritas
{
  "sumber_data": "20 Juli 2026",
  "jumlah_desa_dianalisis": 371,
  "distribusi_zona": {
    "Kemiskinan Tinggi": 136,
    "Kemiskinan Rendah": 126,
    "Kemiskinan Sedang": 109
  },
  "distribusi_jenis_masalah": {
    "Ekonomi": 278,
    "Administratif/Cakupan": 93
  },
  "top5_desa_prioritas": [
    {
      "KECAMATAN": "Kikim Selatan",
      "KELURAHAN": "Beringin Janggut",
      "skor_topsis_ahp": 0.613425
    },
    {
      "KECAMATAN": "Muarapayang",
      "KELURAHAN": "Bandu Agung",
      "skor_topsis_ahp": 0.600643
    },
    {
      "KECAMATAN": "Jarai",
      "KELURAHAN": "Lubuk Saung",
      "skor_topsis_ahp": 0.543371
    },
    {
      "KECAMATAN": "Tanjungtebat",
      "KELURAHAN": "Padang Perigi",
      "skor_topsis_ahp": 0.52036
    },
    {
      "KECAMATAN": "Kikim Selatan",
      "KELURAHAN": "Keban Raya",
      "skor_topsis_ahp": 0.503846
    }
  ],
  "anomali": "Talang Jawa (Kec. Lahat) — perlu survei lapa

## 7. Ringkasan Hasil Akhir

Notebook ini menghasilkan **peta prioritas desa** yang menyatukan clustering (zona),
TOPSIS+AHP (skor & peringkat), dan aturan rekomendasi (jenis masalah → tindakan).

**Output final (`output/spk/peta_prioritas/`):**
- `peta_prioritas_final.csv` — tabel lengkap 371 desa (prioritas, zona, skor, rekomendasi)
- `top15_desa_prioritas.png` — 15 desa teratas
- `prioritas_per_kecamatan.png` — agregasi antar-kecamatan
- `anomali_data.csv` — Talang Jawa (perlu survei)
- `ringkasan_peta_prioritas.json`

Inilah keluaran yang dapat diserahkan/didemokan ke Dinas Sosial Kab. Lahat.